In [1]:
import tensorflow as tf
import sys
sys.path.append('../')
from models.layers import PositionalEmbedding

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
dummy_batch = tf.constant([
    [5, 12, 67, 0],
    [9, 45, 0,  0]
])

vocab_size = 1000
d_model = 256

pos_emb_layer = PositionalEmbedding(vocab_size=vocab_size, d_model=d_model)

output_tensor = pos_emb_layer(dummy_batch)

print("Shape dữ liệu ban đầu:", dummy_batch.shape) 
print("Shape sau khi qua layer:", output_tensor.shape) 

mask = pos_emb_layer.compute_mask(dummy_batch)
print(mask.numpy())

Shape dữ liệu ban đầu: (2, 4)
Shape sau khi qua layer: (2, 4, 256)
[[ True  True  True False]
 [ True  True False False]]


In [4]:
import tensorflow as tf
import sys
sys.path.append('../')
from models.attention import MultiHeadAttention

mha_layer = MultiHeadAttention(d_model=256, num_heads=8)

batch_size = 3
seq_len = 50
d_model = 256

X_dummy = tf.random.normal((batch_size, seq_len, d_model))
print(f"Shape đầu vào X: {X_dummy.shape}")
output, attention_weights = mha_layer(q=X_dummy, k=X_dummy, v=X_dummy, mask=None)

print(f"Shape của Output (Sau khi trộn): {output.shape}") 
print(f"Shape của Bảng điểm Attention: {attention_weights.shape}")

Shape đầu vào X: (3, 50, 256)
Shape của Output (Sau khi trộn): (3, 50, 256)
Shape của Bảng điểm Attention: (3, 8, 50, 50)


In [5]:
from models.encoder import EncoderLayer

sample_encoder_layer = EncoderLayer(d_model=256, num_heads=8, dff=1024)
dummy_x = tf.random.normal((3, 50, 256))
output = sample_encoder_layer(x=dummy_x, training=False, mask=None)

print(f"Shape đầu vào: {dummy_x.shape}")
print(f"Shape đầu ra:  {output.shape}")

Shape đầu vào: (3, 50, 256)
Shape đầu ra:  (3, 50, 256)


In [6]:
from models.encoder import Encoder 

sample_encoder = Encoder(
    num_layers=6, 
    d_model=256, 
    num_heads=8, 
    dff=1024, 
    vocab_size=10000,
    rate=0.1
)

dummy_x = tf.random.uniform((3, 50), minval=1, maxval=8000, dtype=tf.int32)
zero_pad_mask = tf.cast(tf.random.uniform((3, 50)) > 0.2, tf.int32) 
dummy_x = dummy_x * zero_pad_mask 

output = sample_encoder(x=dummy_x, training=False)

print(f"Shape đầu vào (Ma trận ID thô): {dummy_x.shape}")
print(f"Shape đầu ra (Sự thấu hiểu 256 chiều): {output.shape}")

E:\Lib\site-packages\keras\src\layers\layer.py:970: UserWarning: Layer 'multi_head_attention_2' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
E:\Lib\site-packages\keras\src\layers\layer.py:970: UserWarning: Layer 'multi_head_attention_3' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
E:\Lib\site-packages\keras\src\layers\layer.py:970: UserWarning: Layer 'multi_head_attention_4' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
E:\Lib\site-packages\kera

Shape đầu vào (Ma trận ID thô): (3, 50)
Shape đầu ra (Sự thấu hiểu 256 chiều): (3, 50, 256)


In [7]:
import tensorflow as tf
from models.decoder import Decoder 

sample_decoder = Decoder(
    num_layers=6, 
    d_model=256, 
    num_heads=8, 
    dff=1024, 
    vocab_size=10000, 
    rate=0.1
)


batch_size = 3
input_seq_len = 50
target_seq_len = 40 

dummy_enc_output = tf.random.normal((batch_size, input_seq_len, 256))
dummy_enc_padding_mask = tf.cast(tf.random.uniform((batch_size, input_seq_len)) > 0.2, tf.bool)
dummy_target_x = tf.random.uniform((batch_size, target_seq_len), minval=1, maxval=8000, dtype=tf.int32)
target_pad_mask = tf.cast(tf.random.uniform((batch_size, target_seq_len)) > 0.2, tf.int32)
dummy_target_x = dummy_target_x * target_pad_mask

decoder_output = sample_decoder(
    x=dummy_target_x, 
    enc_output=dummy_enc_output, 
    training=False, 
    enc_padding_mask=dummy_enc_padding_mask
)

print("KIỂM TRA ĐẦU VÀO")
print(f"Shape ID câu đích (x):             {dummy_target_x.shape}")
print(f"Shape tri thức Encoder:            {dummy_enc_output.shape}")
print(f"Shape Mask Encoder:                {dummy_enc_padding_mask.shape}\n")

print("KIỂM TRA ĐẦU RA")
print(f"Shape đầu ra của Decoder:          {decoder_output.shape}")

E:\Lib\site-packages\keras\src\layers\layer.py:970: UserWarning: Layer 'multi_head_attention_8' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
E:\Lib\site-packages\keras\src\layers\layer.py:970: UserWarning: Layer 'multi_head_attention_9' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
E:\Lib\site-packages\keras\src\layers\layer.py:970: UserWarning: Layer 'decoder_layer' (of type DecoderLayer) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
E:\Lib\site-packages\keras\src\layers\la

KIỂM TRA ĐẦU VÀO
Shape ID câu đích (x):             (3, 40)
Shape tri thức Encoder:            (3, 50, 256)
Shape Mask Encoder:                (3, 50)

KIỂM TRA ĐẦU RA
Shape đầu ra của Decoder:          (3, 40, 256)


In [9]:
import tensorflow as tf
from models.transformer import Transformer

sample_transformer = Transformer(
    num_layers=4, 
    d_model=128, 
    num_heads=8, 
    dff=512, 
    input_vocab_size=8500,
    tgt_vocab_size=8000,  
    rate=0.1
)

batch_size = 3
inp_seq_len = 50
tar_seq_len = 40

temp_inp = tf.random.uniform((batch_size, inp_seq_len), minval=1, maxval=8500, dtype=tf.int32)
temp_tar = tf.random.uniform((batch_size, tar_seq_len), minval=1, maxval=8000, dtype=tf.int32)

inp_pad_mask = tf.cast(tf.random.uniform((batch_size, inp_seq_len)) > 0.2, tf.int32)
temp_inp = temp_inp * inp_pad_mask

tar_pad_mask = tf.cast(tf.random.uniform((batch_size, tar_seq_len)) > 0.2, tf.int32)
temp_tar = temp_tar * tar_pad_mask
fn_out = sample_transformer(inputs=(temp_inp, temp_tar), training=False)

print("HÔNG SỐ ĐẦU VÀO")
print(f"Shape câu gốc (Input):       {temp_inp.shape}")
print(f"Shape câu đích (Target):     {temp_tar.shape}\n")

print("THÔNG SỐ ĐẦU RA")
print(f"Shape dự đoán (Output):      {fn_out.shape}")

E:\Lib\site-packages\keras\src\layers\layer.py:970: UserWarning: Layer 'multi_head_attention_20' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
E:\Lib\site-packages\keras\src\layers\layer.py:970: UserWarning: Layer 'encoder_layer_7' (of type EncoderLayer) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
E:\Lib\site-packages\keras\src\layers\layer.py:970: UserWarning: Layer 'encoder_layer_8' (of type EncoderLayer) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
E:\Lib\site-packages\keras\src\layers\layer.py:970

HÔNG SỐ ĐẦU VÀO
Shape câu gốc (Input):       (3, 50)
Shape câu đích (Target):     (3, 40)

THÔNG SỐ ĐẦU RA
Shape dự đoán (Output):      (3, 40, 8000)
